# diaph3dt: walkthrough

This notebook runs the pipeline step by step on one case and shows the intermediate results:

1. **Tracking** of each 2D+t plane with CoTracker3 (multi-anchor, medoid consensus, island removal);
2. **Anchoring** of the 3D trajectories on the inspiratory template;
3. **Temporal assembly** of the planes (acquisition delay between planes);
4. **Deformation** of the template at each instant;
5. **Evaluation** against a 3D breath-hold segmentation not used for the reconstruction.

By default it runs on a small **synthetic** case (two domes moving vertically, three sagittal planes),
with CoTracker replaced by a tracker read from the segmentation, so that it runs on a CPU in a few minutes.
Set `USE_SYNTHETIC = False` and the paths below to run it on your own data (see the README for the file layout).

In [ ]:
import os, sys
sys.path[:0] = [os.path.abspath('..'), os.path.abspath('../tests')]

import numpy as np
import matplotlib.pyplot as plt

from diaph3dt.io import find_planes, case_file
from diaph3dt.tracking import track_plane
from diaph3dt.template import load_mesh, top_surface_faces
from diaph3dt.anchoring import anchor_plane
from diaph3dt.timeline import plane_time_offset
from diaph3dt.reconstruction import ReconSettings, reconstruct_case
from diaph3dt.viewer import load_points
from diaph3dt.deformation import PointDeformer
from diaph3dt.evaluation import EvalSettings, evaluate_case
from diaph3dt.summary import best_params

In [ ]:
USE_SYNTHETIC = True
OUT_DIR = 'demo_out'

if USE_SYNTHETIC:
    from synthetic import make_dataset, segmentation_tracker, SUBJECT, VIEW
    SEG_DIR, IMG_DIR = make_dataset('demo_data')
    MODEL, DEVICE, TRACKER = None, 'cpu', segmentation_tracker
else:
    from diaph3dt.tracking import load_cotracker_model, run_cotracker
    SEG_DIR, IMG_DIR = 'path/to/segmentations', 'path/to/images'
    SUBJECT, VIEW = 'SUBJ01', 'D'
    MODEL, DEVICE = load_cotracker_model(device='cuda')
    TRACKER = run_cotracker

planes = find_planes(SEG_DIR, IMG_DIR, SUBJECT, VIEW)
print('planes:', planes)

## 1. Tracking of one plane

Query points are placed on the medial line of the segmentation at several anchor frames spread over the
sequence; each anchor is tracked over the whole sequence. The medoid tracking is kept, then points whose
motion is inconsistent with their neighbours (islands) are excluded. `phis` is the respiratory level of each
frame (0 = inspiratory breath-hold, 1 = expiratory breath-hold).

In [ ]:
track = track_plane(SEG_DIR, IMG_DIR, SUBJECT, VIEW, planes[0], MODEL, DEVICE,
                    n_candidates=5, tracker=TRACKER)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, t in zip(axes[:2], (track['anchor'], int(np.nanargmax([p if p is not None else np.nan for p in track['phis']])))):
    ax.imshow(track['imgs'][t], cmap='gray')
    pts = track['tracks_rc'][t][~track['island_mask']]
    ax.scatter(pts[:, 1], pts[:, 0], c=np.arange(len(pts)), cmap='hsv', s=8)
    ax.set_title(f'frame {t}'); ax.axis('off')
axes[2].plot([p if p is not None else np.nan for p in track['phis']], '.-')
axes[2].set_xlabel('frame'); axes[2].set_ylabel('phi'); axes[2].set_title('respiratory level')
plt.tight_layout()

## 2. Anchoring on the inspiratory template

Each tracked point is mapped to the scanner frame; its 3D trajectory crosses the upper surface of the
inspiratory mesh, which gives its rest position on the template. Frames with a CoTracker confidence below 0.8
are dropped.

In [ ]:
from diaph3dt.io import mask_path, planes_geometry
geom = planes_geometry(SEG_DIR, SUBJECT, VIEW, planes)
mesh_verts, mesh_faces = load_mesh(mask_path(SEG_DIR, SUBJECT, 'inspi'), VIEW, planes_geom=geom)
anchor_faces = top_surface_faces(mesh_verts, mesh_faces)
control_points, diag = anchor_plane(track, mesh_verts, anchor_faces,
                                    t_offset=plane_time_offset(planes[0]))

ax = plt.figure(figsize=(7, 6)).add_subplot(projection='3d')
top = np.unique(anchor_faces)
ax.scatter(*mesh_verts[top].T, s=1, c='lightgray')
for tr in diag['trajectories']:
    ax.plot(*np.array(tr['pts3d']).T, lw=0.8)
    ax.scatter(*tr['anchor'], c='k', s=8)
ax.set_xlabel('R'); ax.set_ylabel('A'); ax.set_zlabel('S'); ax.set_title('trajectories and anchors');

## 3. Full reconstruction of the case

`reconstruct_case` runs steps 1-2 on every plane of the view, assembles the planes in time
(frame t of plane p at t + p/3) and writes the control points, the filtered breath-hold masks and three
interactive viewers (`tracking.html`, `anchoring.html`, `reconstruction.html`).

In [ ]:
case = reconstruct_case(SEG_DIR, IMG_DIR, OUT_DIR, SUBJECT, VIEW, MODEL, DEVICE,
                        settings=ReconSettings(n_candidates=5), tracker=TRACKER)
print(sorted(os.listdir(case)))

## 4. Deformation of the template

At each instant, the displacements of the observed control points (current position minus rest position)
drive the deformation of the template. Here, a thin-plate spline at the most expiratory instant, compared
with the expiratory breath-hold surface (sagittal cut through the middle plane).

In [ ]:
verts, faces = load_mesh(case_file(case, 'inspi'), VIEW, prefiltered=True, top_only=True)
expi_v, _ = load_mesh(case_file(case, 'expi'), VIEW, prefiltered=True, top_only=True)
rest, frames, _ = load_points(case_file(case, 'points'))
fr = max(frames, key=lambda f: f['phi_mean'] if f['phi_mean'] is not None else -1)
ok = ~np.isnan(fr['points'][:, 0])
diag = np.linalg.norm(rest[ok].max(0) - rest[ok].min(0))
deformed = PointDeformer(rest[ok], fr['points'][ok], kind='tps', noise=0.01 * diag).deform(verts)  # TPS, reg = 0.01 (paper)

x0 = np.median(rest[:, 0])
plt.figure(figsize=(7, 4))
for v, lab in ((verts, 'template (inspiratory)'), (deformed, f'deformed, t={fr["t"]:.2f}'), (expi_v, 'expiratory breath-hold')):
    sel = np.abs(v[:, 0] - x0) < 1
    plt.scatter(v[sel, 1], v[sel, 2], s=3, label=lab)
plt.xlabel('A (mm)'); plt.ylabel('S (mm)'); plt.legend(); plt.title(f'sagittal cut x = {x0:.0f} mm');

## 5. Evaluation

Every method is evaluated at the instant t* where the tracked points are closest to the reference
breath-hold surface (chosen before any deformation), with the parameters selected on the tuning
cohort of the paper (`params/paper_params.json`). NSD@3 mm: fraction of the reference within 3 mm
of the reconstruction; s_asd: average symmetric surface distance (mm).

On this synthetic case the dome rises more at the back than at the front and flattens, so a rigid
or affine fit cannot follow it exactly, as in vivo.

In [ ]:
import json
with open('../params/paper_params.json') as f:
    paper = {m: b['params'] for m, b in json.load(f).items()}
rows = []
evaluate_case(SEG_DIR, IMG_DIR, case, SUBJECT, VIEW, EvalSettings(fixed=paper), rows.append)
best = best_params([{k: (float(v) if isinstance(v, (int, float)) else v) for k, v in r.items()} for r in rows],
                   target='normal')
print(f"{'method':12s} {'params':22s} {'NSD@3':>6s} {'ASSD':>6s}")
for m in ['none', 'rigid', 'affine', 'rbf', 'rbf_affine', 'tps', 'biharm']:
    print(f"{m:12s} {best[m]['params']:22s} {best[m]['nsd3']:6.3f} {best[m]['s_asd']:6.2f}")

The viewers can be opened in a browser: `demo_out/<subject>_<view>/reconstruction.html` (template deformation
with live choice of the method, kernel scale and regularisation), `tracking.html` and `anchoring.html`.